# Multi SQL Time Series

Quick notebook to plot multiple SQL queries as a shared time series.
Each query returns one series; all series are normalized and visualized automatically.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import psycopg
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ['POSTGRES_HOST'],
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    user=os.environ['POSTGRES_USER'],
    password=os.environ['POSTGRES_PASSWORD'],
    dbname=os.environ['POSTGRES_DATABASE'],
    sslmode=os.getenv('PGSSLMODE', 'require'),
)

def q(sql: str, **params) -> pd.DataFrame:
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

print('Ready for SQL time series plots')

## 1) Define word patterns

Set the words/patterns in a Python list.
The SQL query stays the same and is generated automatically for each pattern.

In [ ]:
WORD_PATTERNS = [
    "loop engineering",
    "graph engineering",
    # "ai",
    # "robotics",
]

BASE_SQL = '''
select day, words, sources, frequencies
from ngrams_summary
where words like '{pattern}'
order by day desc
'''

SQL_QUERIES = [BASE_SQL.format(pattern=pattern) for pattern in WORD_PATTERNS]

SQL_QUERIES

In [ ]:
DATE_CANDIDATES = ['day', 'date', 'timestamp', 'ts']
METRIC_COLUMN = 'sources'
FREQUENCY_COLUMN = 'frequencies'
WORD_COLUMN = 'words'

def normalize_timeseries(df: pd.DataFrame, fallback_series: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['day', 'sources', 'frequencies', 'word', 'series'])

    cols = {c.lower(): c for c in df.columns}

    day_col = None
    for c in DATE_CANDIDATES:
        if c in cols:
            day_col = cols[c]
            break
    if day_col is None:
        raise ValueError(f'No time column found in {list(df.columns)}')

    source_col = cols.get(METRIC_COLUMN)
    if source_col is None:
        raise ValueError(
            f"Column '{METRIC_COLUMN}' not found in {list(df.columns)}"
        )

    frequency_col = cols.get(FREQUENCY_COLUMN)
    word_col = cols.get(WORD_COLUMN)

    keep_cols = [day_col, source_col]
    if frequency_col is not None:
        keep_cols.append(frequency_col)
    if word_col is not None:
        keep_cols.append(word_col)

    out = df[keep_cols].copy()

    rename_map = {day_col: 'day', source_col: 'sources'}
    if frequency_col is not None:
        rename_map[frequency_col] = 'frequencies'
    if word_col is not None:
        rename_map[word_col] = 'word'
    out = out.rename(columns=rename_map)

    if 'frequencies' not in out.columns:
        out['frequencies'] = out['sources']
    if 'word' not in out.columns:
        out['word'] = fallback_series

    out['day'] = pd.to_datetime(out['day'], errors='coerce')
    out = out.dropna(subset=['day'])
    out['sources'] = pd.to_numeric(out['sources'], errors='coerce')
    out['frequencies'] = pd.to_numeric(out['frequencies'], errors='coerce')
    out['word'] = out['word'].astype(str)
    out = out.dropna(subset=['sources', 'frequencies'])

    out = out.groupby(['day', 'word'], as_index=False).agg(
        {'sources': 'sum', 'frequencies': 'sum'}
    )
    out['series'] = out['word']
    return out.sort_values(['word', 'day'])

frames = []
for i, sql in enumerate(SQL_QUERIES, start=1):
    raw = q(sql)
    norm = normalize_timeseries(raw, f'query_{i}')
    frames.append(norm)

all_series = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['day', 'sources', 'frequencies', 'word', 'series'])
all_series.tail(20)

In [ ]:
if all_series.empty:
    print('No data found.')
else:
    freq = all_series['frequencies'].fillna(0).clip(lower=0)
    size_min, size_max = 7, 28

    if freq.max() == freq.min():
        all_series = all_series.copy()
        all_series['marker_size'] = (size_min + size_max) / 2
    else:
        all_series = all_series.copy()
        all_series['marker_size'] = (
            size_min
            + (freq - freq.min()) * (size_max - size_min) / (freq.max() - freq.min())
        )

    fig = go.Figure()
    for series_name, part in all_series.groupby('series', sort=False):
        fig.add_trace(
            go.Scatter(
                x=part['day'],
                y=part['sources'],
                mode='lines+markers',
                name=series_name,
                marker=dict(
                    size=part['marker_size'],
                    sizemode='diameter',
                    line=dict(width=0.5, color='white'),
                ),
                customdata=part[['frequencies']],
                hovertemplate='Series=%{fullData.name}<br>Day=%{x|%Y-%m-%d}<br>Sources=%{y}<br>Frequencies=%{customdata[0]}<extra></extra>',
            )
        )

    fig.update_layout(
        height=520,
        xaxis_title='',
        yaxis_title='Sources',
        title='Time series comparison across multiple SQL queries',
    )
    fig.show()

In [ ]:
# Optional: 7-day smoothing for noisy signals
if all_series.empty:
    print('No data for rolling mean.')
else:
    smooth = all_series.sort_values('day').copy()
    smooth['rolling_7d_sources'] = (
        smooth.groupby('series')['sources']
        .transform(lambda s: s.rolling(7, min_periods=1).mean())
    )

    fig = px.line(
        smooth,
        x='day',
        y='rolling_7d_sources',
        color='series',
        title='7-day rolling mean per series',
    )
    fig.update_layout(height=520, xaxis_title='', yaxis_title='Rolling Mean Sources')
    fig.show()